<a href="https://colab.research.google.com/github/prince127-web/GenAI/blob/main/pdfchatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U openai sentence-transformers chromadb langchain-text-splitters pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204

In [2]:
import os
from openai import OpenAI
from google.colab import userdata

# Get OpenAI API key from Colab Secrets
api_key = userdata.get("openai")

if not api_key:
    raise ValueError(
        "OpenAI API key not found. Add 'OpenAI_API_Key' "
        "to Colab Secrets and enable Notebook access."
    )

client = OpenAI(api_key=api_key)

print("OpenAI client initialized successfully.")

OpenAI client initialized successfully.


In [7]:
from google.colab import files
from pypdf import PdfReader

print("Please upload your PDF:")

uploaded = files.upload()

pdf_text = ""

for filename in uploaded.keys():

    if filename.lower().endswith(".pdf"):

        reader = PdfReader(filename)

        for page_number, page in enumerate(reader.pages, start=1):

            text = page.extract_text()

            if text:
                pdf_text += (
                    f"\n--- Page {page_number} ---\n"
                    f"{text}"
                )

        print(
            f"Loaded: {filename} "
            f"({len(reader.pages)} pages)"
        )

if not pdf_text.strip():
    raise ValueError("No readable PDF text found.")

print("PDF text extracted successfully.")

Please upload your PDF:


Saving ACAP-NOTICE-2026-2.pdf to ACAP-NOTICE-2026-2.pdf
Loaded: ACAP-NOTICE-2026-2.pdf (2 pages)
PDF text extracted successfully.


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_text(pdf_text)

print(f"Created {len(chunks)} chunks.")

Created 11 chunks.


In [9]:
from sentence_transformers import SentenceTransformer
import chromadb

print("Loading embedding model...")

embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Creating embeddings...")

embeddings = embedder.encode(
    chunks,
    show_progress_bar=True
).tolist()

# Create ChromaDB client
chroma_client = chromadb.Client()

# Delete old collection if it exists
try:
    chroma_client.delete_collection(
        name="pdf_rag_collection"
    )
except Exception:
    pass

# Create collection
collection = chroma_client.create_collection(
    name="pdf_rag_collection"
)

# IDs for chunks
ids = [f"chunk_{i}" for i in range(len(chunks))]

# Add data
collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=ids
)

print(f"Successfully indexed {len(chunks)} chunks.")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Successfully indexed 11 chunks.


In [12]:
def ask_pdf(question, number_of_chunks=5):

    # Create embedding for the question
    question_embedding = embedder.encode(
        [question]
    )[0].tolist()

    # Search ChromaDB
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=number_of_chunks
    )

    # Get relevant PDF chunks
    documents = results["documents"][0]

    # Combine chunks into context
    context = "\n\n".join(
        [
            f"--- PDF Section {i + 1} ---\n{doc}"
            for i, doc in enumerate(documents)
        ]
    )

    # OpenAI prompt
    prompt = f"""
You are a helpful PDF assistant.

Answer the user's question using ONLY the information
contained in the PDF context below.

Rules:
1. Do not make up information.
2. Do not use outside knowledge.
3. If the answer is not available in the PDF,
   say: "I couldn't find this information in the uploaded PDF."
4. Give a clear and concise answer.

PDF CONTEXT:
{context}

USER QUESTION:
{question}
"""

    # Call OpenAI
    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )

    return response.output_text

In [13]:
question = "What is this document about?"

answer = ask_pdf(question)

print("Assistant:")
print(answer)

Assistant:
This document is a circular providing instructions for admissions to institutional-level (Management Quota) seats and vacant seats remaining after the CAP rounds for technical education courses for the academic year 2026–27. It also specifies requirements for preparing merit lists and submitting daily reports of attended and admitted students.


In [14]:
question = "What are the important dates mentioned in the document?"

print(ask_pdf(question))

The important dates mentioned are:

- **Circular date:** 02 September 2026  
- **Academic year:** 2026–27  

The date fields for the daily attendance reports are blank in the document.
